# Profiling Messy CSVs with Dataprof

In data engineering, you often receive CSV files that contain mixed data types, missing values, and corrupted text. This exercise shows how to use `dataprof` to swiftly detect and summarize these issues.

In [ ]:
# Run this cell first to install dependencies in Colab
!pip install pandas dataprof

## Generate Messy Dummy Data

Let's create a terribly formatted CSV to simulate wild, unclean data. Notice the arbitrary date formats, missing objects, and mixed numeric/text fields.

In [ ]:
import pandas as pd
import numpy as np

# Build the corrupted dataset elements
messy_data = {
    'user_id': ['101', '102', '103', 'UNKNOWN', '105', '106', '107', None, '109', '110'],
    'age': [25, 30, 'thirty-five', 40, np.nan, 22, -5, 29, 31, 899],
    'signup_date': ['2023-01-01', '2023-01-02', 'not_a_date', '2023/15/01', '2023-01-05', '01-06-2023', pd.NaT, '2023-01-08', '20230109', '2023-01-10'],
    'score': [8.5, 9.0, np.nan, 7.5, 8.0, 9.5, 6.0, 7.0, np.nan, 8.2]
}

df_messy = pd.DataFrame(messy_data)
df_messy.to_csv("messy_data.csv", index=False)
print("Created 'messy_data.csv'!")

display(df_messy)

## Profile the Data

We'll load the literal string versions of this CSV (which is how pandas usually falls back on messy data without schema pre-validation) and then use `dataprof` to expose the anomalies.

In [ ]:
import dataprof as dp
import json

# 1. Load the data exactly as it appears in the wild
df_raw = pd.read_csv("messy_data.csv", dtype=str)

# 2. Hand it off to dataprof to crunch the underlying anomalies
report = dp.profile(df_raw)

# 3. Extract key metrics
print("--- DATA PROFILE SUMMARY ---")
print(f"Total Rows Diagnosed: {report.rows}")

print("\n--- CRITICAL COLUMN INSIGHTS ---")
for col_name, col_profile in report.column_profiles.items():
    print(f"\nColumn: '{col_name}'")
    print(f"  - Missing Value Rate: {col_profile.null_count} out of {report.rows}")
    
    # Check the underlying data distribution and unique occurrences
    if hasattr(col_profile, 'min_value') and col_profile.min_value is not None:
         print(f"  - Range detected: {col_profile.min_value} to {col_profile.max_value}")

# 4. Write full schema out for automated CI/CD checks next time
print("\nSaving report to JSON for further inspection...")
report.save("messy_report.json")